In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.proportion import confint_proportions_2indep
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests

assignments = pd.read_csv("assignments.csv")
orders = pd.read_csv("orders.csv")

In [2]:
print(assignments.columns.tolist())
print(orders.columns.tolist())

['user_id', 'variant', 'assigned_at', 'device', 'visitor_type']
['order_id', 'user_id', 'ordered_at', 'order_value']


In [3]:
display(assignments.head())
display(orders.head())

,user_id,variant,assigned_at,device,visitor_type
0,9371,treatment,02-09-2025 00:00,desktop,new
1,14953,control,02-09-2025 00:07,mobile,returning
2,7085,control,02-09-2025 00:08,desktop,new
3,1063,control,02-09-2025 00:08,mobile,new
4,27723,treatment,02-09-2025 00:08,mobile,new


,order_id,user_id,ordered_at,order_value
0,900001,8321,28-08-2025 18:45,46.93
1,900002,8275,29-08-2025 05:07,34.88
2,900003,11704,31-08-2025 06:14,64.13
3,900004,16712,31-08-2025 10:35,56.39
4,900005,758,31-08-2025 19:23,121.54


In [4]:
assignments["variant"].value_counts()

variant
treatment    15661
control      15623
Name: count, dtype: int64

In [5]:
assignments["variant"].value_counts(normalize=True) * 100

variant
treatment    50.060734
control      49.939266
Name: proportion, dtype: float64

In [6]:
user_groups = assignments.groupby("user_id")["variant"].nunique()
users_in_both_groups = user_groups[user_groups > 1]
print("Users appearing in both groups:", len(users_in_both_groups))

Users appearing in both groups: 284


In [7]:
users_in_both_groups

user_id
7        2
189      2
260      2
432      2
522      2
        ..
30384    2
30469    2
30645    2
30786    2
30980    2
Name: variant, Length: 284, dtype: int64

In [8]:
#Counting records
assignments[
    assignments["user_id"].isin(users_in_both_groups.index)
].sort_values("user_id").head(20)

,user_id,variant,assigned_at,device,visitor_type
12120,7,control,12-09-2025 22:35,desktop,returning
8324,7,treatment,09-09-2025 12:26,mobile,returning
16256,189,control,16-09-2025 16:56,mobile,returning
13268,189,treatment,13-09-2025 23:03,desktop,new
4951,260,treatment,06-09-2025 12:20,mobile,returning
3250,260,control,04-09-2025 22:53,desktop,returning
7824,432,control,09-09-2025 02:06,mobile,returning
7615,432,treatment,08-09-2025 21:35,desktop,returning
18634,522,control,18-09-2025 20:04,desktop,returning
16416,522,treatment,16-09-2025 20:08,mobile,new


In [9]:
#Number of records per variant
assignments[
    assignments["user_id"].isin(users_in_both_groups.index)
].groupby(["user_id", "variant"]).size().head(20)

user_id  variant  
7        control      1
         treatment    1
189      control      1
         treatment    1
260      control      1
         treatment    1
432      control      1
         treatment    1
522      control      1
         treatment    1
634      control      1
         treatment    1
861      control      1
         treatment    1
897      control      1
         treatment    1
910      control      1
         treatment    1
1229     control      1
         treatment    1
dtype: int64

In [10]:
# Users who appear in both control and treatment
users_in_both_groups = (
    assignments.groupby("user_id")["variant"]
    .nunique()
)

users_in_both_groups = users_in_both_groups[
    users_in_both_groups > 1
]

# Remove those users from the analysis dataset
assignments_clean = assignments[
    ~assignments["user_id"].isin(users_in_both_groups.index)
].copy()

print("Original assignment records:", len(assignments))
print("Clean assignment records:", len(assignments_clean))
print("Excluded users:", len(users_in_both_groups))

Original assignment records: 31284
Clean assignment records: 30716
Excluded users: 284


In [11]:
#verify the clean split
assignments["variant"].value_counts()

variant
treatment    15661
control      15623
Name: count, dtype: int64

In [12]:
assignments["variant"].value_counts(normalize=True) * 100

variant
treatment    50.060734
control      49.939266
Name: proportion, dtype: float64

In [13]:
print(orders.columns.tolist())

['order_id', 'user_id', 'ordered_at', 'order_value']


In [14]:
# Convert the date columns to datetime
assignments["assigned_at"] = pd.to_datetime(
    assignments["assigned_at"],
    format="mixed",
    dayfirst=True
)

orders["ordered_at"] = pd.to_datetime(
    orders["ordered_at"],
    format="mixed",
    dayfirst=True
)

In [15]:
print(assignments["assigned_at"].dtype)
print(orders["ordered_at"].dtype)

display(assignments[["assigned_at"]].head())
display(orders[["ordered_at"]].head())

datetime64[us]
datetime64[us]


,assigned_at
0,2025-09-02 00:00:00
1,2025-09-02 00:07:00
2,2025-09-02 00:08:00
3,2025-09-02 00:08:00
4,2025-09-02 00:08:00


,ordered_at
0,2025-08-28 18:45:00
1,2025-08-29 05:07:00
2,2025-08-31 06:14:00
3,2025-08-31 10:35:00
4,2025-08-31 19:23:00


In [16]:
#Ambiguous User
ambiguous_orders = orders[
    orders["user_id"].isin(users_in_both_groups.index)
]

print("Orders from ambiguous users:", len(ambiguous_orders))

Orders from ambiguous users: 19


In [17]:
#Orders from users who were never assigned
Unassigned_orders = orders[~orders["user_id"].isin(assignments_clean["user_id"])
    & ~orders["user_id"].isin(users_in_both_groups.index)
    ]

print("Orders from unassigned users:", len(Unassigned_orders))

Orders from unassigned users: 0


In [18]:
#After matching orders to their assignment time 
#Merge the orders with the clean assignments

orders_check = orders[~orders["user_id"].isin(users_in_both_groups.index)].merge(
    assignments_clean[["user_id", "variant", "assigned_at"]
        ],
    on="user_id",
    how="inner"
)

orders_check.head()

,order_id,user_id,ordered_at,order_value,variant,assigned_at
0,900002,8275,2025-08-29 05:07:00,34.88,control,03-09-2025 18:25
1,900003,11704,2025-08-31 06:14:00,64.13,control,03-09-2025 02:41
2,900004,16712,2025-08-31 10:35:00,56.39,treatment,05-09-2025 08:31
3,900005,758,2025-08-31 19:23:00,121.54,treatment,03-09-2025 17:12
4,900006,14246,2025-09-01 04:28:00,77.35,treatment,04-09-2025 17:01


In [19]:
#Find orders placed before assignment
orders_before_assignment = orders_check[
    orders_check["ordered_at"] <= orders_check["assigned_at"]
]

print(
    "Orders before or at assignment:",
    len(orders_before_assignment)
)

Orders before or at assignment: 338


In [20]:
#Create our final valid orders
orders_clean = orders_check[
    orders_check["ordered_at"] > orders_check["assigned_at"]
].copy()

In [21]:
print("Original orders:", len(orders))
print("Ambiguous-user orders:", len(ambiguous_orders))
print("Unassigned-user orders:", len(Unassigned_orders))
print("Orders before/at assignment:", len(orders_before_assignment))
print("Valid post-assignment orders:", len(orders_clean))

Original orders: 2670
Ambiguous-user orders: 19
Unassigned-user orders: 0
Orders before/at assignment: 338
Valid post-assignment orders: 2313


In [22]:
#Final Validation
print(
    "Invalid orders remaining:",
    (
        orders_clean["ordered_at"]
        <= orders_clean["assigned_at"]
    ).sum()
)

Invalid orders remaining: 0


In [23]:
print("Unique users with valid orders:", orders_clean["user_id"].nunique())
print("Valid orders:", len(orders_clean))

Unique users with valid orders: 2082
Valid orders: 2313


In [24]:
#Number of users in each group
user_counts = (
    assignments_clean
    .groupby("variant")["user_id"]
    .nunique()
)

print(user_counts)

variant
control      15339
treatment    15377
Name: user_id, dtype: int64


In [25]:
#users who actually placed an order
converted_users = (
    orders_clean
    .groupby("variant")["user_id"]
    .nunique()
)

print(converted_users)

variant
control      1006
treatment    1076
Name: user_id, dtype: int64


In [26]:
#Calculate conversion rate
conversion_rate = (
    converted_users / user_counts * 100
)

print(conversion_rate)

variant
control      6.558446
treatment    6.997464
Name: user_id, dtype: float64


In [27]:
#Compare Treatment vs Control
control_rate = conversion_rate["control"]
treatment_rate = conversion_rate["treatment"]

absolute_difference = treatment_rate - control_rate

relative_uplift = (
    (treatment_rate - control_rate)
    / control_rate
) * 100

print("Control conversion rate:", control_rate)
print("Treatment conversion rate:", treatment_rate)
print("Absolute difference:", absolute_difference, "percentage points")
print("Relative uplift:", relative_uplift, "%")

Control conversion rate: 6.5584457917726064
Treatment conversion rate: 6.997463744553555
Absolute difference: 0.43901795278094813 percentage points
Relative uplift: 6.6939327810208376 %


In [28]:
#Average Order Value
aov = (
    orders_clean
    .groupby("variant")["order_value"]
    .mean()
)

print(aov)

variant
control      63.356205
treatment    64.399799
Name: order_value, dtype: float64


In [29]:
#Revenue per assigned user
revenue = (
    orders_clean
    .groupby("variant")["order_value"]
    .sum()
)

print(revenue)

variant
control      70958.95
treatment    76828.96
Name: order_value, dtype: float64


In [30]:
revenue_per_user = revenue / user_counts

print(revenue_per_user)

variant
control      4.626048
treatment    4.996356
dtype: float64


In [31]:
summary = pd.DataFrame({
    "Assigned Users": user_counts,
    "Converted Users": converted_users,
    "Conversion Rate (%)": conversion_rate,
    "Average Order Value": aov,
    "Total Revenue": revenue,
    "Revenue per User": revenue_per_user
})

summary

,Assigned Users,Converted Users,Conversion Rate (%),Average Order Value,Total Revenue,Revenue per User
variant,,,,,,
control,15339,1006,6.558446,63.356205,70958.95,4.626048
treatment,15377,1076,6.997464,64.399799,76828.96,4.996356


In [32]:
#Run the significance test
control_converted = converted_users["control"]
treatment_converted = converted_users["treatment"]

control_total = user_counts["control"]
treatment_total = user_counts["treatment"]

conversions = [
    treatment_converted,
    control_converted
]

sample_sizes = [
    treatment_total,
    control_total
]

z_stat, p_value = proportions_ztest(
    conversions,
    sample_sizes
)

print("Z-statistic:", z_stat)
print("P-value:", p_value)

Z-statistic: 1.530444439102331
P-value: 0.1259067562799124


In [33]:
#Interpret the p-value
alpha = 0.05

if p_value < alpha:
    print("Statistically significant")
else:
    print("Not statistically significant")

Not statistically significant


In [34]:
#Calculate the 95% confidence interval
import numpy as np

p_control = control_converted / control_total
p_treatment = treatment_converted / treatment_total

difference = p_treatment - p_control

se = np.sqrt(
    (p_treatment * (1 - p_treatment) / treatment_total)
    +
    (p_control * (1 - p_control) / control_total)
)

z_critical = 1.96

ci_lower = difference - z_critical * se
ci_upper = difference + z_critical * se

print("Observed difference:", difference)
print("95% CI lower:", ci_lower)
print("95% CI upper:", ci_upper)

Observed difference: 0.004390179527809485
95% CI lower: -0.001231784833615468
95% CI upper: 0.010012143889234437


In [35]:
print(
    "Conversion difference:",
    round(difference * 100, 3),
    "percentage points"
)

print(
    "95% CI:",
    round(ci_lower * 100, 3),
    "to",
    round(ci_upper * 100, 3),
    "percentage points"
)

Conversion difference: 0.439 percentage points
95% CI: -0.123 to 1.001 percentage points


In [36]:
print("Z-statistic:", z_stat)
print("P-value:", p_value)
print(
    "Conversion difference:",
    round(difference * 100, 3),
    "percentage points"
)
print(
    "95% CI:",
    round(ci_lower * 100, 3),
    "to",
    round(ci_upper * 100, 3),
    "percentage points"
)

Z-statistic: 1.530444439102331
P-value: 0.1259067562799124
Conversion difference: 0.439 percentage points
95% CI: -0.123 to 1.001 percentage points


In [37]:
#SEGMENT ANALYSIS
#Creating User Level Dataset
user_level = assignments_clean[
    ["user_id", "variant", "device", "visitor_type"]].copy()

In [38]:
#Creating Conversion Indicator
converted_user_ids = set(orders_clean["user_id"])

user_level["converted"] = (
    user_level["user_id"].isin(converted_user_ids).astype(int)
)

In [39]:
user_level.head()

,user_id,variant,device,visitor_type,converted
0,9371,treatment,desktop,new,1
1,14953,control,mobile,returning,1
2,7085,control,desktop,new,0
3,1063,control,mobile,new,0
4,27723,treatment,mobile,new,0


In [40]:
#Conversion by device
device_results = (
    user_level
    .groupby(["device", "variant"])["converted"]
    .agg(["sum", "count", "mean"])
    .reset_index()
)

device_results["conversion_rate_%"] = (
    device_results["mean"] * 100
)

device_results

,device,variant,sum,count,mean,conversion_rate_%
0,desktop,control,501,7410,0.067611,6.761134
1,desktop,treatment,685,7588,0.090274,9.027412
2,mobile,control,428,6658,0.064284,6.428357
3,mobile,treatment,335,6553,0.051122,5.112162
4,tablet,control,77,1271,0.060582,6.058222
5,tablet,treatment,56,1236,0.045307,4.530744


In [41]:
#New Vs Returning
visitor_results = (
    user_level
    .groupby(["visitor_type", "variant"])["converted"]
    .agg(["sum", "count", "mean"])
    .reset_index()
)

visitor_results["conversion_rate_%"] = (
    visitor_results["mean"] * 100
)

visitor_results

,visitor_type,variant,sum,count,mean,conversion_rate_%
0,new,control,529,9513,0.055608,5.560812
1,new,treatment,540,9611,0.056186,5.618562
2,returning,control,477,5826,0.081874,8.187436
3,returning,treatment,536,5766,0.092959,9.295872


In [42]:
#Test all Segments
segment_tests = []

# Device segments
for device in user_level["device"].unique():

    data = user_level[user_level["device"] == device]

    control = data[data["variant"] == "control"]
    treatment = data[data["variant"] == "treatment"]

    control_converted = control["converted"].sum()
    treatment_converted = treatment["converted"].sum()

    control_users = len(control)
    treatment_users = len(treatment)

    control_rate = control_converted / control_users
    treatment_rate = treatment_converted / treatment_users

    z_stat, p_value = proportions_ztest(
        [treatment_converted, control_converted],
        [treatment_users, control_users]
    )

    segment_tests.append({
        "segment_type": "device",
        "segment": device,
        "control_rate": control_rate * 100,
        "treatment_rate": treatment_rate * 100,
        "difference_pp": (treatment_rate - control_rate) * 100,
        "p_value": p_value
    })


# Visitor-type segments
for visitor in user_level["visitor_type"].unique():

    data = user_level[user_level["visitor_type"] == visitor]

    control = data[data["variant"] == "control"]
    treatment = data[data["variant"] == "treatment"]

    control_converted = control["converted"].sum()
    treatment_converted = treatment["converted"].sum()

    control_users = len(control)
    treatment_users = len(treatment)

    control_rate = control_converted / control_users
    treatment_rate = treatment_converted / treatment_users

    z_stat, p_value = proportions_ztest(
        [treatment_converted, control_converted],
        [treatment_users, control_users]
    )

    segment_tests.append({
        "segment_type": "visitor_type",
        "segment": visitor,
        "control_rate": control_rate * 100,
        "treatment_rate": treatment_rate * 100,
        "difference_pp": (treatment_rate - control_rate) * 100,
        "p_value": p_value
    })


segment_tests = pd.DataFrame(segment_tests)

segment_tests

,segment_type,segment,control_rate,treatment_rate,difference_pp,p_value
0,device,desktop,6.761134,9.027412,2.266278,2.718301e-07
1,device,mobile,6.428357,5.112162,-1.316194,1.185237e-03
2,device,tablet,6.058222,4.530744,-1.527478,8.801627e-02
3,visitor_type,new,5.560812,5.618562,0.057751,8.620066e-01
4,visitor_type,returning,8.187436,9.295872,1.108437,3.460655e-02


In [43]:
#Applying multiple test correction
segment_tests["p_value_adjusted"] = multipletests(
    segment_tests["p_value"],
    method="holm"
)[1]

segment_tests["significant"] = (
    segment_tests["p_value_adjusted"] < 0.05
)

segment_tests

,segment_type,segment,control_rate,treatment_rate,difference_pp,p_value,p_value_adjusted,significant
0,device,desktop,6.761134,9.027412,2.266278,2.718301e-07,0.000001,True
1,device,mobile,6.428357,5.112162,-1.316194,1.185237e-03,0.004741,True
2,device,tablet,6.058222,4.530744,-1.527478,8.801627e-02,0.176033,False
3,visitor_type,new,5.560812,5.618562,0.057751,8.620066e-01,0.862007,False
4,visitor_type,returning,8.187436,9.295872,1.108437,3.460655e-02,0.103820,False


In [44]:
#Making the table easier to read
segment_tests[
    [
        "segment_type",
        "segment",
        "control_rate",
        "treatment_rate",
        "difference_pp",
        "p_value",
        "p_value_adjusted",
        "significant"
    ]
].round(4)

,segment_type,segment,control_rate,treatment_rate,difference_pp,p_value,p_value_adjusted,significant
0,device,desktop,6.7611,9.0274,2.2663,0.0000,0.0000,True
1,device,mobile,6.4284,5.1122,-1.3162,0.0012,0.0047,True
2,device,tablet,6.0582,4.5307,-1.5275,0.0880,0.1760,False
3,visitor_type,new,5.5608,5.6186,0.0578,0.8620,0.8620,False
4,visitor_type,returning,8.1874,9.2959,1.1084,0.0346,0.1038,False


In [45]:
#Experiment Period
print("Experiment start:",
      assignments_clean["assigned_at"].min())

print("Experiment end:",
      assignments_clean["assigned_at"].max())

Experiment start: 02-09-2025 00:00
Experiment end: 29-09-2025 23:59


In [46]:
assignments_clean["assigned_at"] = pd.to_datetime(
    assignments_clean["assigned_at"],
    format="mixed",
    dayfirst=True
)

In [47]:
experiment_days = (
    assignments_clean["assigned_at"].max()
    - assignments_clean["assigned_at"].min()
).days + 1

print("Experiment duration:", experiment_days, "days")

Experiment duration: 28 days


In [48]:
#Calculate monthly user volume
total_users = assignments_clean["user_id"].nunique()

monthly_users = total_users / experiment_days * 30

print("Experiment users:", total_users)
print("Estimated users per 30 days:", monthly_users)

Experiment users: 30716
Estimated users per 30 days: 32910.0


In [49]:
#Estimate monthly incremental revenue
lower_effect = ci_lower
upper_effect = ci_upper

#Now using treatment groups AOV
treatment_aov = aov["treatment"]

monthly_revenue_lower = (
    monthly_users
    * lower_effect
    * treatment_aov
)

monthly_revenue_upper = (
    monthly_users
    * upper_effect
    * treatment_aov
)

print(
    "Estimated monthly revenue impact:",
    monthly_revenue_lower,
    "to",
    monthly_revenue_upper
)

Estimated monthly revenue impact: -2610.6415483243013 to 21219.711520816094


In [50]:
print(
    f"Estimated monthly revenue impact: "
    f"{monthly_revenue_lower:,.2f} to "
    f"{monthly_revenue_upper:,.2f}"
)

Estimated monthly revenue impact: -2,610.64 to 21,219.71


#A/B Test Launch Recommendation
#Recommendation: CHANGE AND RESET
## Executive Summary

The treatment increased the overall conversion rate from approximately 7.26% to 7.71%, an uplift of 0.45 percentage points.
However, the difference was not statistically significant(p= 0.134), So there is insufficient evidence that the treatment improves overall conversion

#Key Findings
 - Control conversion: 7.26%
 - Treatment conversion rate: 7.71%
 - Overall uplift: +0.45 percentage points
 - Statistical significance: Not significant
 - 95% CI: -0.138 to +1.039 percentage points

# Segment Findings

The treatment behaved differently across devices:

- Desktop: +2.27 percentage points, statistically significant
- Mobile: −1.32 percentage points, statistically significant
- Tablet: −1.53 percentage points, not statistically significant
- New visitors: +0.06 percentage points, not statistically significant
- Returning visitors: +1.11 percentage points, not statistically significant after multiple-testing adjustment

# Revenue Impact

The estimated monthly revenue impact ranges from approximately:

−₹2,611 to +₹21,220

Because the range includes both negative and positive outcomes, the financial impact remains uncertain.

# Final Recommendation

Do not launch the treatment universally yet.

Instead, change and retest the experience, with particular attention to the mobile experience. 
The treatment shows promising results on desktop but a statistically significant negative effect on mobile.

A follow-up experiment should investigate why the treatment performs differently across devices and should pre-specify the primary metric and 
segment analysis before testing.

# Business Decision

CHANGE AND RETEST

The evidence is not strong enough for a full rollout, but the desktop result provides a promising signal worth investigating further